# v32.1 unified one-run with no-skip advocate fallback

Purpose: fix v32 where model loaded but `n_generated=0` because dataset/premise alignment failed.  
This notebook keeps v30.1 locked baseline and tries targeted advocate generation for YNU cases. If no advocates are generated, it fails loud before rerank/full.

Outputs:
- `v32_1_a_pool_oracle_audit_summary.json`, `v32_1_a_pool_oracle_audit_cases.json`
- `v32_1_b_advocate_candidates.json`
- `v32_1_standard_preds.json`, `v32_1_standard_summary.json`
- `v32_1_full_preds.json`, `v32_1_full_summary.json`
- `v32_1_risk_report.json`

In [ ]:
# ============================================================
# Config
# ============================================================
import os, re, json, math, random, textwrap, gc, glob, sys, subprocess
from pathlib import Path
from collections import Counter, defaultdict

SEED = 20260610
random.seed(SEED)

LOCKED_V30_MACRO = 0.5934206145879246
LOCKED_V30_DIFFS = {71,109,125}
LABELS = ['A','B','C','D','Yes','No','Unknown']
YNU = {'Yes','No','Unknown'}
BANKED = {71,109,125}

CANDIDATE_DIRS = [
    Path('/kaggle/working'),
    Path('/kaggle/input/datasets/yixuanisthebest/v2333333'),
    Path('/kaggle/input/datasets/nguyenminhtric/test-pipeline'),
    Path('/mnt/data'),
]
print('CANDIDATE_DIRS:', [str(x) for x in CANDIDATE_DIRS])

In [ ]:
# ============================================================
# IO + metric helpers
# ============================================================
def find_file(name, required=True):
    for d in CANDIDATE_DIRS:
        p = d / name
        if p.exists(): return p
    # tolerate browser duplicate names: foo(1).json
    stem = Path(name).stem; suf = Path(name).suffix
    for d in CANDIDATE_DIRS:
        if d.exists():
            hits = sorted(d.glob(stem + '*' + suf))
            if hits: return hits[0]
    if required:
        raise FileNotFoundError(f'Required file not found: {name}; looked in {[str(d) for d in CANDIDATE_DIRS]}')
    return None

def load_json(name, required=True):
    p = find_file(name, required)
    if p is None: return None, None
    with open(p, 'r', encoding='utf-8') as f: return json.load(f), p

def save_json(obj, name):
    p = Path('/kaggle/working')/name if Path('/kaggle/working').exists() else Path(name)
    with open(p, 'w', encoding='utf-8') as f: json.dump(obj, f, ensure_ascii=False, indent=2)
    print('SAVED', p)
    return p

def get_pred(row): return row.get('pred', row.get('prediction'))
def set_pred(row, label):
    row['pred'] = label
    row['prediction'] = label

def normalize_label(v):
    if v is None: return None
    s = str(v).strip()
    if not s: return None
    up=s.upper()
    if up in 'ABCD': return up
    t=s.title()
    return t if t in ['Yes','No','Unknown'] else None

def compute_metrics(rows):
    y_true=[r['gold'] for r in rows]
    y_pred=[get_pred(r) for r in rows]
    n=len(rows)
    acc=sum(a==b for a,b in zip(y_true,y_pred))/n
    per={}
    for lab in LABELS:
        tp=sum((a==lab and b==lab) for a,b in zip(y_true,y_pred))
        fp=sum((a!=lab and b==lab) for a,b in zip(y_true,y_pred))
        fn=sum((a==lab and b!=lab) for a,b in zip(y_true,y_pred))
        support=sum(a==lab for a in y_true)
        pc=sum(b==lab for b in y_pred)
        prec=tp/(tp+fp) if tp+fp else 0.0
        rec=tp/(tp+fn) if tp+fn else 0.0
        f1=2*prec*rec/(prec+rec) if prec+rec else 0.0
        per[lab]={'precision':prec,'recall':rec,'f1':f1,'support':support,'pred_count':pc}
    macro=sum(per[l]['f1'] for l in LABELS)/len(LABELS)
    weighted=sum(per[l]['f1']*per[l]['support'] for l in LABELS)/n
    mc_macro=sum(per[l]['f1'] for l in ['A','B','C','D'])/4
    ynu_macro=sum(per[l]['f1'] for l in ['Yes','No','Unknown'])/3
    return {'n':n,'acc':acc,'macro_f1':macro,'weighted_f1':weighted,'mc_macro':mc_macro,'ynu_macro':ynu_macro,'per_label':per,'gold_count':dict(Counter(y_true)),'pred_count':dict(Counter(y_pred))}

def diffs(a,b):
    return [i for i,(x,y) in enumerate(zip(a,b)) if get_pred(x)!=get_pred(y)]

def save_and_verify(rows, summary, pred_name, summary_name):
    pred_path=save_json(rows, pred_name)
    summ_path=save_json(summary, summary_name)
    re_rows,_=load_json(pred_name)
    re_metrics=compute_metrics(re_rows)
    sm = summary.get('metrics') or summary.get('candidate_metrics') or summary.get('new_metrics') or {}
    if sm:
        assert abs(re_metrics['macro_f1'] - sm['macro_f1']) < 1e-10, (re_metrics['macro_f1'], sm['macro_f1'])
        assert abs(re_metrics['mc_macro'] - sm['mc_macro']) < 1e-10
    return pred_path, summ_path, re_metrics

def brief_metrics(m):
    return f"acc={m['acc']:.6f} macro={m['macro_f1']:.10f} weighted={m['weighted_f1']:.6f} MC={m['mc_macro']:.6f} YNU={m['ynu_macro']:.6f}"

In [ ]:
# ============================================================
# Load and verify v30.1 locked baseline
# ============================================================
v30_rows, v30_path = load_json('v30_1_full_preds.json')
v30_summary, v30_sum_path = load_json('v30_1_full_summary.json', required=False)
base_rows, base_path = load_json('v27_standard_preds.json', required=False)

v30_metrics = compute_metrics(v30_rows)
print('Loaded v30:', v30_path)
print('v30 metrics:', brief_metrics(v30_metrics))
assert abs(v30_metrics['macro_f1'] - LOCKED_V30_MACRO) < 1e-8, 'v30.1 macro mismatch'
if base_rows:
    d=set(diffs(base_rows, v30_rows))
    print('v30 diffs vs v27:', sorted(d))
    assert d == LOCKED_V30_DIFFS, f'v30.1 diffs mismatch: {d}'
print('baseline v30.1 VERIFIED')

In [ ]:
# ============================================================
# Candidate pool loading with positional alignment
# ============================================================
raw_candidates, cand_path = load_json('v23_val_candidates.json')
print('Loaded candidates:', cand_path, 'type=', type(raw_candidates).__name__)
FINAL_RE = re.compile(r'Final Answer\s*[:\-]?\s*(Yes|No|Unknown|[ABCD])\b', re.I)

def cand_answer(c):
    if isinstance(c, str):
        m=FINAL_RE.findall(c)
        return normalize_label(m[-1]) if m else None
    if isinstance(c, dict):
        for k in ('answer','final_answer','final','pred','prediction','ans','label','verdict_answer'):
            if k in c and c[k] is not None:
                lab=normalize_label(c[k])
                if lab: return lab
        for k in ('text','raw','output','explanation','completion','response','generation','content'):
            if k in c and c[k]:
                m=FINAL_RE.findall(str(c[k]))
                if m: return normalize_label(m[-1])
    return None

def cand_text(c):
    if isinstance(c, str): return c
    if isinstance(c, dict):
        parts=[]
        for k in ('text','raw','output','explanation','completion','response','generation','content'):
            if k in c and c[k]: parts.append(str(c[k]))
        if parts: return '\n'.join(parts)
        return json.dumps(c, ensure_ascii=False)[:4000]
    return str(c)

def candidate_records(raw):
    if isinstance(raw, list): return raw
    if isinstance(raw, dict):
        for k in ('data','samples','records','items'):
            if isinstance(raw.get(k), list): return raw[k]
        recs=[]
        for k,v in raw.items():
            if isinstance(v,dict):
                vv=dict(v); vv.setdefault('idx', int(k) if str(k).isdigit() else k); recs.append(vv)
            elif isinstance(v,list): recs.append({'idx': int(k) if str(k).isdigit() else k, 'candidates': v})
        return recs
    raise AssertionError(type(raw).__name__)

def rec_cands(rec):
    if isinstance(rec, list): return rec
    if isinstance(rec, dict):
        for k in ('candidates','cands','outputs','generations','samples','completions','responses','predictions'):
            if isinstance(rec.get(k), list): return rec[k]
        if cand_answer(rec): return [rec]
    return None

def build_pool(raw, preds):
    recs=candidate_records(raw)
    print('candidate records:', len(recs), 'first keys:', sorted(recs[0].keys())[:20] if isinstance(recs[0],dict) else None)
    pool={}; qmatch=0
    norm=lambda q: re.sub(r'\s+',' ',str(q).strip().lower())[:500]
    for pos,rec in enumerate(recs):
        if pos>=len(preds): break
        cl=rec_cands(rec)
        if cl is None: continue
        if isinstance(rec,dict) and rec.get('question') and preds[pos].get('question') and norm(rec['question'])==norm(preds[pos]['question']): qmatch+=1
        pool[pos]=[{'ans':cand_answer(c),'text':cand_text(c),'raw_type':type(c).__name__} for c in cl]
    ynu_n=sum(not r.get('is_mc') for r in preds)
    ynu_cov=sum((not preds[i].get('is_mc')) and i in pool for i in range(len(preds)))
    print(f'pool built for {len(pool)}/{len(preds)} rows; exact question matches={qmatch}; YNU covered={ynu_cov}/{ynu_n}; sample sizes={Counter(len(v) for v in pool.values())}')
    assert len(pool)>=0.95*len(preds), 'pool coverage too low'
    return pool

pool=build_pool(raw_candidates, v30_rows)

In [ ]:
# ============================================================
# Optional dataset alignment for premises; robust fallback if failed
# ============================================================
def flatten_records(x):
    out=[]
    if isinstance(x, list):
        for it in x: out.extend(flatten_records(it))
    elif isinstance(x, dict):
        # record if it has a question-like field
        if any(k in x for k in ('question','query','prompt')):
            out.append(x)
        for k,v in x.items():
            if isinstance(v,(list,dict)) and k not in ('candidates','outputs','generations','responses'):
                out.extend(flatten_records(v))
    return out

def load_optional_dataset():
    names=['repaired_dataset_v5.json','v5_repaired_dataset.json','v5_val.json','val.json','dataset.json','data.json']
    for name in names:
        p=find_file(name, required=False)
        if p:
            try:
                raw=json.load(open(p,encoding='utf-8'))
                recs=flatten_records(raw)
                print('optional dataset loaded:',p,'flat records=',len(recs))
                return recs,p
            except Exception as e:
                print('optional dataset failed:', p, e)
    print('No optional dataset/premise file found; generation will use fallback context.')
    return [], None

def rec_question(r):
    if not isinstance(r,dict): return ''
    return r.get('question') or r.get('query') or r.get('prompt') or ''

def rec_premises(r):
    if not isinstance(r,dict): return ''
    keys=['premises','context','facts','rules','input','problem','source_text']
    parts=[]
    for k in keys:
        v=r.get(k)
        if not v: continue
        if isinstance(v,list):
            parts.append('\n'.join(f'{j+1}. {x}' for j,x in enumerate(v)))
        else:
            parts.append(str(v))
    return '\n'.join(parts)

def align_dataset(preds):
    recs,p=load_optional_dataset()
    aligned={}
    if not recs: return aligned, {'path':None,'aligned':0,'mode':'none'}
    norm=lambda q: re.sub(r'\s+',' ',str(q).strip().lower())[:500]
    byq={norm(rec_question(r)):r for r in recs if rec_question(r)}
    qmatch=0
    for i,row in enumerate(preds):
        rq=norm(row.get('question',''))
        if rq in byq:
            aligned[i]=byq[rq]; qmatch+=1
    if qmatch < 0.5*len(preds) and len(recs)==len(preds):
        aligned={i:recs[i] for i in range(len(preds))}
        mode='positional_fallback'
    else:
        mode='question_match'
    with_prem=sum(bool(rec_premises(r)) for r in aligned.values())
    print(f'dataset aligned {len(aligned)}/{len(preds)} mode={mode} with_premises={with_prem}')
    return aligned, {'path':str(p),'aligned':len(aligned),'mode':mode,'with_premises':with_prem}

dataset_aligned, dataset_diag = align_dataset(v30_rows)

In [ ]:
# ============================================================
# v32.1_a: pool-oracle audit on wrong YNU cases
# ============================================================
wrong_cases=[]; fam=Counter()
for i,row in enumerate(v30_rows):
    pred=get_pred(row); gold=row['gold']
    if row.get('is_mc') or pred==gold: continue
    if pred not in YNU and gold not in YNU: continue
    cands=pool.get(i,[])
    ans=[c['ans'] for c in cands]
    parse_fails=sum(a is None for a in ans)
    vote=dict(Counter(a for a in ans if a))
    if parse_fails==len(cands): family='D_parse_fail_all'
    elif gold in vote: family='A_gold_in_pool'
    else: family='B_no_gold_candidate'
    fam[family]+=1
    wrong_cases.append({'idx':i,'gold':gold,'pred':pred,'family':family,'vote':vote,'n_cands':len(cands),'parse_fails':parse_fails,'question':row.get('question','')[:220]})
summary_a={'version':'v32_1_a_pool_oracle_audit','families':dict(fam),'routing':'v32.1 no-skip advocate generation + verifier rerank','n_wrong':len(wrong_cases),'dataset_diag':dataset_diag}
save_json(summary_a,'v32_1_a_pool_oracle_audit_summary.json')
save_json(wrong_cases,'v32_1_a_pool_oracle_audit_cases.json')
print('v32.1_a families:', dict(fam), 'n_wrong=', len(wrong_cases))

In [ ]:
# ============================================================
# v32.1_b: advocate generation with no-skip fallback context
# ============================================================
RUN_GENERATION = True  # set False to skip GPU generation and force rollback diagnostic
MAX_NEW_TOKENS = 420
TEMPERATURE = 0.2
TOP_P = 0.9
TARGET_LIMIT = None  # optional int for smoke test

# Target: YNU rows with pred No/UNPARSEABLE, excluding banked v30 flips. Includes correct No too for cautious verifier context.
target_indices=[]
for i,row in enumerate(v30_rows):
    pred=get_pred(row)
    if row.get('is_mc'): continue
    if i in BANKED: continue
    if pred in ('No','UNPARSEABLE'):
        target_indices.append(i)
if TARGET_LIMIT: target_indices=target_indices[:TARGET_LIMIT]
print('target cases:', len(target_indices), target_indices[:20])

def format_old_candidates(cands, max_chars=3000):
    parts=[]
    for j,c in enumerate(cands[:5]):
        parts.append(f"Candidate {j}: answer={c.get('ans')}\n{c.get('text','')[:900]}")
    return '\n\n'.join(parts)[:max_chars]

def build_context(idx):
    row=v30_rows[idx]
    parts=[]
    rec=dataset_aligned.get(idx)
    prem=rec_premises(rec) if rec else ''
    if prem.strip(): parts.append('Premises / context from dataset:\n'+prem[:5000])
    # Always include fallback fields, even when premises exist, to avoid zero context.
    parts.append('Question:\n'+str(row.get('question','')))
    if row.get('explanation'):
        parts.append('Current selected explanation and final answer:\n'+str(row.get('explanation',''))[:2500])
    old=format_old_candidates(pool.get(idx,[]), max_chars=3500)
    if old: parts.append('Existing candidate pool (may be biased):\n'+old)
    ctx='\n\n'.join(p for p in parts if p and str(p).strip())
    return ctx[:9000]

def make_prompt(idx, stance):
    row=v30_rows[idx]
    return f"""You are a strict logical verifier for an educational QA task.

Task: Write an ADVOCATE argument for the target final answer: {stance}.
But you must be honest: if the target answer is not supported by the premises/context, say VERDICT: INVALID.
If it is supported, say VERDICT: VALID.

Required output schema:
Reasoning: <concise premise-grounded chain>
Supporting Premises: [numbers if available, otherwise []]
Final Answer: {stance}
VERDICT: VALID or INVALID

Context:
{build_context(idx)}
"""

def parse_verdict(text):
    m=re.findall(r'VERDICT\s*[:\-]?\s*(VALID|INVALID)', text, flags=re.I)
    return m[-1].upper() if m else None

def has_citation(text):
    return bool(re.search(r'Supporting Premises\s*[:\-]?\s*\[[^\]]*\d', text or '', flags=re.I))

advocates={'__meta__':{'load_error':None,'n_generated':0,'n_skipped':0,'skipped':[],'load_mode':None,'target_indices':target_indices}}
if not RUN_GENERATION:
    advocates['__meta__']['load_error']='RUN_GENERATION=False'
else:
    try:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        from peft import PeftModel
        print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'n_gpu', torch.cuda.device_count())
        # Discover model/adapter. Override via env if needed.
        BASE_MODEL=os.environ.get('BASE_MODEL')
        ADAPTER=os.environ.get('ADAPTER')
        def discover_adapter():
            hits=[]
            roots=['/kaggle/input','/mnt/data']
            for root in roots:
                for p in Path(root).rglob('adapter_config.json'):
                    hits.append(str(p.parent))
            return hits[0] if hits else None
        def discover_base():
            # prefer env; otherwise common Qwen path names under kaggle input with config and model weights
            hits=[]
            for root in ['/kaggle/input','/mnt/data']:
                for p in Path(root).rglob('config.json'):
                    parent=str(p.parent)
                    if 'checkpoint' in parent.lower(): continue
                    if any((Path(parent)/x).exists() for x in ['model.safetensors','pytorch_model.bin']) or list(Path(parent).glob('model-*.safetensors')):
                        hits.append(parent)
            return hits[0] if hits else None
        if ADAPTER is None: ADAPTER=discover_adapter()
        if BASE_MODEL is None: BASE_MODEL=discover_base()
        print('BASE_MODEL=',BASE_MODEL)
        print('ADAPTER=',ADAPTER)
        assert BASE_MODEL, 'BASE_MODEL not found; set os.environ["BASE_MODEL"]'
        assert ADAPTER, 'ADAPTER not found; set os.environ["ADAPTER"]'
        dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
        load_attempts=[]
        # Try 4bit only if bitsandbytes is compatible.
        try:
            import bitsandbytes as bnb
            qconf=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=dtype,bnb_4bit_use_double_quant=True)
            load_attempts.append(('bnb4', dict(device_map='auto', quantization_config=qconf)))
        except Exception as e:
            print('bitsandbytes unavailable/incompatible:', repr(e))
        load_attempts.append(('sharded_dtype', dict(device_map='auto', dtype=dtype)))
        model=None; last_err=None
        for mode,kw in load_attempts:
            try:
                print('MODEL LOAD ATTEMPT:', mode)
                model=AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kw)
                model=PeftModel.from_pretrained(model, ADAPTER)
                tok=AutoTokenizer.from_pretrained(ADAPTER)
                if tok.pad_token is None: tok.pad_token=tok.eos_token
                model.eval(); advocates['__meta__']['load_mode']=mode; break
            except Exception as e:
                print('MODEL LOAD FAILED:', mode, repr(e)); last_err=repr(e); model=None
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
        if model is None: raise RuntimeError(last_err)
        def generate_one(prompt):
            messages=[{'role':'user','content':prompt}]
            if hasattr(tok, 'apply_chat_template'):
                text=tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            else:
                text=prompt
            inputs=tok(text, return_tensors='pt', truncation=True, max_length=8192)
            inputs={k:v.to(model.device) for k,v in inputs.items()}
            with torch.no_grad():
                out=model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE, top_p=TOP_P, pad_token_id=tok.eos_token_id)
            gen=tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            return gen
        for idx in target_indices:
            ctx=build_context(idx)
            if not ctx.strip():
                advocates['__meta__']['n_skipped']+=1; advocates['__meta__']['skipped'].append(idx); continue
            advocates[str(idx)]={}
            for stance in ['Yes','No','Unknown']:
                prompt=make_prompt(idx, stance)
                try:
                    txt=generate_one(prompt)
                    advocates[str(idx)][stance]={'text':txt,'verdict':parse_verdict(txt),'cited':has_citation(txt),'final_answer':stance}
                except Exception as e:
                    advocates[str(idx)][stance]={'error':repr(e),'text':'','verdict':None,'cited':False,'final_answer':stance}
            advocates['__meta__']['n_generated']+=1
            if advocates['__meta__']['n_generated'] % 5 == 0: print('generated', advocates['__meta__']['n_generated'])
    except Exception as e:
        advocates['__meta__']['load_error']=repr(e)
        print('GENERATION FAILED:', repr(e))

save_json(advocates,'v32_1_b_advocate_candidates.json')
print('advocate meta:', advocates['__meta__'])
# v32.1 change: fail-loud if model loaded but all target rows skipped/generated zero.
if advocates['__meta__'].get('load_error') is None:
    assert advocates['__meta__']['n_generated'] > 0, 'Advocate generation produced zero rows despite no load_error; context alignment/fallback broken. STOP.'

In [ ]:
# ============================================================
# v32.1_standard: advocate verifier rerank
# ============================================================
advocates,_ = load_json('v32_1_b_advocate_candidates.json')
new_rows=[dict(r) for r in v30_rows]
base_metrics=v30_metrics
flips=[]; decision_log=[]

meta=advocates.get('__meta__',{}) if isinstance(advocates,dict) else {}
if meta.get('load_error') or meta.get('n_generated',0)==0:
    selection_status='ROLLBACK_v30_1'
    reason=f"No usable advocates: load_error={meta.get('load_error')} n_generated={meta.get('n_generated')}"
else:
    for idx_s,adv in advocates.items():
        if idx_s == '__meta__': continue
        try: idx=int(idx_s)
        except: continue
        if idx in BANKED: continue
        row=new_rows[idx]
        if row.get('is_mc'): continue
        old=get_pred(row)
        if old not in ('No','UNPARSEABLE'): continue
        yes=adv.get('Yes',{}); no=adv.get('No',{}); unk=adv.get('Unknown',{})
        yes_valid = yes.get('verdict')=='VALID' and yes.get('cited')
        no_invalid = no.get('verdict')=='INVALID'
        no_valid = no.get('verdict')=='VALID' and no.get('cited')
        unk_valid = unk.get('verdict')=='VALID' and unk.get('cited')
        action=None; newpred=old; rule=None
        # S2 asymmetric advocate: Yes valid, No invalid; Unknown not valid.
        if yes_valid and no_invalid and not unk_valid:
            newpred='Yes'; rule='S2_yes_valid_no_invalid'
        # S3 UNPARSEABLE unique valid advocate only.
        if old=='UNPARSEABLE':
            valids=[lab for lab,a in [('Yes',yes),('No',no),('Unknown',unk)] if a.get('verdict')=='VALID' and a.get('cited')]
            if len(valids)==1:
                newpred=valids[0]; rule='S3_unparseable_unique_valid'
        if newpred != old:
            set_pred(row,newpred)
            flips.append({'idx':idx,'old':old,'new':newpred,'gold':row['gold'],'rule':rule})
            decision_log.append({'idx':idx,'old':old,'new':newpred,'gold':row['gold'],'rule':rule,'yes':{k:yes.get(k) for k in ['verdict','cited']},'no':{k:no.get(k) for k in ['verdict','cited']},'unknown':{k:unk.get(k) for k in ['verdict','cited']}})
    cand_metrics=compute_metrics(new_rows)
    gain=cand_metrics['macro_f1']-base_metrics['macro_f1']
    no_drop=base_metrics['per_label']['No']['f1']-cand_metrics['per_label']['No']['f1']
    unk_drop=base_metrics['per_label']['Unknown']['f1']-cand_metrics['per_label']['Unknown']['f1']
    collapse= any(cand_metrics['per_label'][l]['pred_count'] < 0.5*base_metrics['per_label'][l]['pred_count'] for l in ['Yes','No','Unknown'])
    # bootstrap over per-row correctness delta as a rough stability check
    deltas=[]
    for oldrow,newrow in zip(v30_rows,new_rows):
        deltas.append((get_pred(newrow)==newrow['gold']) - (get_pred(oldrow)==oldrow['gold']))
    if flips:
        bs=[]
        for _ in range(1000):
            samp=[random.choice(deltas) for _ in deltas]
            bs.append(sum(samp)/len(samp))
        p_positive=sum(x>0 for x in bs)/len(bs); mean=sum(bs)/len(bs); ci_lo=sorted(bs)[25]; ci_hi=sorted(bs)[975]
    else:
        p_positive=0.0; mean=0.0; ci_lo=0.0; ci_hi=0.0
    selected = (gain > 1e-10 and cand_metrics['macro_f1'] > LOCKED_V30_MACRO and no_drop <= 0.05 and unk_drop <= 0.0 and not collapse and cand_metrics['mc_macro']==base_metrics['mc_macro'] and len(flips)<=15 and p_positive>=0.80)
    selection_status='SELECT_V32_1' if selected else 'ROLLBACK_v30_1'
    reason='passes gates' if selected else 'gates failed'
    if not selected:
        new_rows=[dict(r) for r in v30_rows]
        cand_metrics=base_metrics

metrics=compute_metrics(new_rows)
gates={'gain':metrics['macro_f1']-base_metrics['macro_f1'],'unk_drop':base_metrics['per_label']['Unknown']['f1']-metrics['per_label']['Unknown']['f1'],'no_drop':base_metrics['per_label']['No']['f1']-metrics['per_label']['No']['f1'],'collapse':False,'mc_frozen':metrics['mc_macro']==base_metrics['mc_macro'],'selected':selection_status}
summary_std={'version':'v32_1_standard_advocate_verifier_rerank','selection_status':selection_status,'reason':reason,'metrics':metrics,'baseline_v30_1':base_metrics,'n_flips':len(flips),'flips':flips,'decision_log':decision_log,'bootstrap':{'mean':mean if 'mean' in locals() else 0.0,'ci_lo':ci_lo if 'ci_lo' in locals() else 0.0,'ci_hi':ci_hi if 'ci_hi' in locals() else 0.0,'p_positive':p_positive if 'p_positive' in locals() else 0.0},'gates':gates,'advocate_meta':meta}
save_and_verify(new_rows, summary_std, 'v32_1_standard_preds.json', 'v32_1_standard_summary.json')
print('v32.1 standard:', selection_status, brief_metrics(metrics), 'flips=', len(flips), 'reason=', reason)

In [ ]:
# ============================================================
# v32.1_full: select or rollback with saved-preds verification
# ============================================================
std_rows,_=load_json('v32_1_standard_preds.json')
std_sum,_=load_json('v32_1_standard_summary.json')
std_metrics=compute_metrics(std_rows)
actual_diffs=set(diffs(v30_rows,std_rows))
expected=set(f['idx'] for f in std_sum.get('flips',[]))
verified = (actual_diffs==expected and abs(std_metrics['macro_f1']-std_sum['metrics']['macro_f1'])<1e-10 and std_metrics['mc_macro']==v30_metrics['mc_macro'])
select = verified and std_sum.get('selection_status')=='SELECT_V32_1' and std_metrics['macro_f1']>LOCKED_V30_MACRO
if select:
    final_rows=std_rows; final_status='SELECT_V32_1'; source='v32_1_standard_verified'; reason='v32.1 standard passed gates and saved preds verify'
else:
    final_rows=[dict(r) for r in v30_rows]; final_status='ROLLBACK_TO_V30_1'; source='ROLLBACK_v30_1'; reason=f"Rollback: verified={verified} status={std_sum.get('selection_status')} saved_macro={std_metrics['macro_f1']:.10f} actual_diffs={sorted(actual_diffs)} expected={sorted(expected)}"
final_metrics=compute_metrics(final_rows)
summary_full={'version':'v32_1_full','selection_status':final_status,'source':source,'reason':reason,'metrics':final_metrics,'per_label':final_metrics['per_label'],'pred_count':final_metrics['pred_count'],'base_metrics':v30_metrics,'actual_diffs_vs_v30_1':sorted(diffs(v30_rows, final_rows))}
save_and_verify(final_rows, summary_full, 'v32_1_full_preds.json','v32_1_full_summary.json')
print('v32.1 full:', final_status, brief_metrics(final_metrics), reason)

In [ ]:
# ============================================================
# Risk report
# ============================================================
final_rows,_=load_json('v32_1_full_preds.json')
final_sum,_=load_json('v32_1_full_summary.json')
final_metrics=compute_metrics(final_rows)
delta={k:final_metrics[k]-v30_metrics[k] for k in ['acc','macro_f1','weighted_f1','mc_macro','ynu_macro']}
# risk heuristic
if final_sum['selection_status'].startswith('ROLLBACK'):
    overfit='LOW_OPERATIONAL_ROLLBACK'
else:
    nf=len(final_sum.get('actual_diffs_vs_v30_1',[]))
    overfit='MEDIUM_VALIDATION_TUNED' if nf<=15 else 'HIGH_TOO_MANY_FLIPS'
underfit='HIGH_CANDIDATE_INSUFFICIENCY' if summary_a['families'].get('B_no_gold_candidate',0)>=15 else 'MEDIUM_SELECTION_GAP'
collapse_labels=[]
for lab in ['Yes','No','Unknown']:
    if final_metrics['per_label'][lab]['pred_count'] < 0.5*v30_metrics['per_label'][lab]['pred_count']:
        collapse_labels.append(lab)
artifact='LOW_VERIFIED_SAVED_PREDS'
rec='SELECT_V32_1' if final_sum['selection_status']=='SELECT_V32_1' else 'ROLLBACK_TO_V30_1'
risk={'version':'v32_1_unified_risk_report','final_status':final_sum['selection_status'],'final_decision':rec,'metrics':{k:final_metrics[k] for k in ['n','acc','macro_f1','weighted_f1','mc_macro','ynu_macro']},'delta_vs_v30_1':delta,'per_label':final_metrics['per_label'],'n_flips':len(final_sum.get('actual_diffs_vs_v30_1',[])),'overfit_risk':overfit,'underfit_risk':underfit,'label_collapse':bool(collapse_labels),'collapse_labels':collapse_labels,'artifact_risk':artifact,'pool_audit':summary_a,'advocate_meta':meta,'recommendation':rec}
save_json(risk,'v32_1_risk_report.json')
print('\n===== FINAL V32.1 REPORT =====')
print('decision:', rec)
print('metrics:', brief_metrics(final_metrics))
print('delta:', {k:round(v,10) for k,v in delta.items()})
print('n_flips:', risk['n_flips'])
print('overfit_risk:', overfit)
print('underfit_risk:', underfit)
print('label_collapse:', bool(collapse_labels), collapse_labels)
print('artifact_risk:', artifact)
print('advocate_meta:', meta)